# NumPy Essentials for Signals & Systems Lab
### CSE 219 — Building blocks you'll reuse in every assignment

This notebook is a reference, not a homework solution. Every operation here is a
**tool** — the assignments (time reversal, even/odd decomposition, time shift,
phase change, time sub-scaling/interpolation) are built by combining these tools.
Nothing here directly solves an assignment task; it teaches the mechanism so you
can write that logic yourself.

**Topics covered:**
1. Building the time/sample axis (`arange`, `linspace`)
2. Indexing & slicing (basic, step, negative)
3. Reversing arrays (core of time reversal)
4. Shifting arrays / padding with zeros (core of time shift)
5. Boolean masking & `np.where` (core of "ignore values outside range")
6. Rounding indices safely (core of time scaling — mapping `t/k` to array positions)
7. Interpolation building blocks (core of `interpolate_signal`)
8. Vectorized math vs. loops (cos/sin signals, phase)
9. Energy & power computations (`sum`, `abs`, `trapz`)
10. Comparing signals (MSE, `np.array_equal`, `np.allclose`)
11. Copy vs. view — a classic bug source


## 1. Building the axis: `arange` vs `linspace`

Every signal starts with an axis. For **continuous-time signals sampled at some
rate**, you'll almost always use `arange` with an explicit step (the sampling
interval), because you need to know exactly what that step is to reverse/shift/scale
later. `linspace` is useful when you know how many *points* you want, not the
step size.

In [1]:
import numpy as np

# arange(start, stop, step) -- stop is EXCLUSIVE
t1 = np.arange(-5, 5, 0.5)      # step = 0.5, this IS your sampling interval dt
print("arange axis  :", t1[:6], "... length =", len(t1))

# linspace(start, stop, num) -- stop is INCLUSIVE, you specify point COUNT
t2 = np.linspace(-5, 5, 21)     # 21 points from -5 to 5 inclusive
print("linspace axis:", t2[:6], "... length =", len(t2))

# Recovering dt from a linspace axis (useful when a function hands you an axis
# and you need to know the sample spacing, e.g. for interpolation)
dt = t2[1] - t2[0]
print("recovered dt:", dt)

arange axis  : [-5.  -4.5 -4.  -3.5 -3.  -2.5] ... length = 20
linspace axis: [-5.  -4.5 -4.  -3.5 -3.  -2.5] ... length = 21
recovered dt: 0.5


**Why this matters for the assignments:** in Assignment 1 and 3 you generate
`t` once and then must produce `x(-t)` or `x(t/k)` from it. If you don't know your
`dt` precisely, your reversed/scaled axis won't line up with the original, and
your plots (or interpolation) will be wrong. Always compute `dt` explicitly
rather than assuming it.

## 2. Indexing & Slicing

Syntax: `arr[start:stop:step]`. `start` is inclusive, `stop` is exclusive,
`step` can be negative.

In [2]:
x = np.arange(10, 20)   # [10, 11, ..., 19]
print("full array   :", x)
print("x[2]         :", x[2])          # single element
print("x[2:5]       :", x[2:5])        # index 2,3,4  (5 excluded)
print("x[:4]        :", x[:4])         # from start to index 3
print("x[4:]        :", x[4:])         # from index 4 to end
print("x[-1]        :", x[-1])         # last element
print("x[-3:]       :", x[-3:])        # last 3 elements
print("x[::2]       :", x[::2])        # every 2nd element
print("x[1:8:3]     :", x[1:8:3])      # start=1, stop=8, step=3

full array   : [10 11 12 13 14 15 16 17 18 19]
x[2]         : 12
x[2:5]       : [12 13 14]
x[:4]        : [10 11 12 13]
x[4:]        : [14 15 16 17 18 19]
x[-1]        : 19
x[-3:]       : [17 18 19]
x[::2]       : [10 12 14 16 18]
x[1:8:3]     : [11 14 17]


### Negative step — the key to time/signal reversal

`arr[::-1]` reverses the whole array. This is the single most important slicing
trick for the time-reversal assignment: reversing the **sample order** is how you
implement `x(-t)` once your axis is symmetric/uniformly spaced.

In [3]:
x = np.array([0, 1, 2, 3, 4, 5])
print("original :", x)
print("reversed :", x[::-1])

# Reversing with an offset
print("reverse skipping last element:", x[-2::-1])   # start at index -2, go backwards

original : [0 1 2 3 4 5]
reversed : [5 4 3 2 1 0]
reverse skipping last element: [4 3 2 1 0]


**Careful:** `arr[::-1]` gives you the values in reverse order, but it does
**not** by itself give you `x(-t)` sampled on your *original* time axis unless
your axis is symmetric about 0 (e.g. `-5` to `5`) and uniformly spaced. If the
axis isn't symmetric, reversing the array reverses the *values*, but you still
need to think about what time value each reversed sample now corresponds to.
This is exactly the kind of detail `time_reverse(...)` needs to handle correctly.

## 3. Shifting arrays: the mechanics behind time shift

A time shift `x[n - n0]` means: the value that used to be at position `n` moves
to position `n + n0`. In array terms, this is a **shift with zero-padding** (values
shifted "out of range" are dropped; new empty slots are filled with 0, since the
signal is assumed 0 outside its defined support).

In [4]:
x = np.array([1, 2, 3, 4, 5])
n0 = 2  # shift amount

# Manual right-shift by n0 (delay): pad zeros on the left, drop last n0 values
shifted_right = np.concatenate([np.zeros(n0), x])[:len(x)]
print("delayed (right shift):", shifted_right)

# Manual left-shift by n0 (advance): drop first n0 values, pad zeros on the right
shifted_left = np.concatenate([x[n0:], np.zeros(n0)])
print("advanced (left shift):", shifted_left)

# np.roll is tempting but WRAPS AROUND instead of zero-padding -- usually WRONG
# for signals with finite support. Shown here so you recognize the pitfall.
print("np.roll (wraps, rarely what you want):", np.roll(x, n0))

delayed (right shift): [0. 0. 1. 2. 3.]
advanced (left shift): [3. 4. 5. 0. 0.]
np.roll (wraps, rarely what you want): [4 5 1 2 3]


**Why this matters:** in Assignment 2, `time_shift_sinusoid` shifts a sinusoid
in its **argument**, i.e. you evaluate `cos(Ω₀(n - n0) + φ)` directly on the axis
rather than shifting array contents — the sinusoid is defined for all `n`, so
there's no "falling off the edge" to worry about, unlike a finite-support signal.
Recognizing *which situation you're in* (shift the array vs. shift inside the
formula) is the key conceptual step.

## 4. Boolean masking & `np.where` — filtering values outside a range

Assignment 3 asks you to **ignore values that go beyond the specified time
range** after scaling. Boolean masks are the clean way to do this.

In [5]:
t = np.linspace(-10, 10, 21)
y = t**2

mask = (t >= -5) & (t <= 5)     # boolean array, same shape as t
print("mask   :", mask)
print("t in range:", t[mask])
print("y in range:", y[mask])

# np.where can either give you the matching INDICES...
idx = np.where(mask)[0]
print("indices where True:", idx)

# ...or act like a vectorized if/else: np.where(condition, if_true, if_false)
y_clipped = np.where(mask, y, np.nan)   # replace out-of-range values with NaN
print("clipped (NaN outside range):", y_clipped)

mask   : [False False False False False  True  True  True  True  True  True  True
  True  True  True  True False False False False False]
t in range: [-5. -4. -3. -2. -1.  0.  1.  2.  3.  4.  5.]
y in range: [25. 16.  9.  4.  1.  0.  1.  4.  9. 16. 25.]
indices where True: [ 5  6  7  8  9 10 11 12 13 14 15]
clipped (NaN outside range): [nan nan nan nan nan 25. 16.  9.  4.  1.  0.  1.  4.  9. 16. 25. nan nan
 nan nan nan]


Two common patterns for "ignore out-of-range values" when plotting:
1. **Filter arrays before plotting**: `plt.plot(t[mask], y[mask])`
2. **Set out-of-range values to `NaN`**: matplotlib simply skips NaNs when drawing
   a line, so the plotted array stays the same length but gaps appear correctly.
Both are valid; (2) is often more convenient when you need `t` and `y` to stay
the same length for further computation (e.g. comparing with another signal
sample-by-sample).

## 5. Rounding indices safely — the heart of time scaling

`y(t) = x(t/k)` means: for every output time `t`, look up the value of `x` at
time `t/k`. But `x` is only defined at *discrete sample points*. So you need to
convert a continuous time value into the **nearest valid array index**. Getting
this rounding right (and handling out-of-bounds indices) is the single trickiest
part of Assignment 3.

In [6]:
t = np.arange(-5, 5, 1.0)     # original sample axis, dt = 1.0
x = np.sin(t)

def time_to_index(t_query, t_axis):
    """Map a continuous time value to the nearest sample index in t_axis."""
    dt = t_axis[1] - t_axis[0]
    idx_float = (t_query - t_axis[0]) / dt     # position as a float
    idx = np.round(idx_float).astype(int)      # round to nearest INTEGER index
    return idx

# Example: where does t=2.4 fall on our axis?
print("float index for t=2.4:", (2.4 - t[0]) / (t[1]-t[0]))
print("rounded index        :", time_to_index(2.4, t))

# IMPORTANT: np.round uses "round half to even" (banker's rounding), not
# always what you expect for .5 cases:
print("np.round(2.5) =", np.round(2.5), "  np.round(3.5) =", np.round(3.5))
# If you need traditional "round half away from zero", implement it explicitly:
def round_half_up(x):
    return np.floor(x + 0.5) if x >= 0 else np.ceil(x - 0.5)
print("round_half_up(2.5) =", round_half_up(2.5))

float index for t=2.4: 7.4
rounded index        : 7
np.round(2.5) = 2.0   np.round(3.5) = 4.0
round_half_up(2.5) = 3.0


**Bounds checking matters too.** After computing `idx`, always check it's a
valid index before using it (`0 <= idx < len(x)`); otherwise you'll get an
`IndexError` or, worse, silently wrap around with negative indexing. This is
exactly why Assignment 3 says to *"ignore values that go beyond the specified
time range"* — some computed indices for `y(t) = x(t/k)` will fall outside the
original sampled range of `x`, and you must detect and exclude those rather than
letting NumPy silently index into the wrong place.

## 6. Interpolation building blocks

Time sub-scaling with `k > 1` compresses the time axis, which means the new
grid asks for values *between* your original samples. Assignment 3 specifically
asks you to average the **nearest left and right neighbors** — not to use a
library interpolator — so here's how to build that averaging manually, plus
`np.interp` shown only as a reference for what "real" linear interpolation gives.

In [7]:
x = np.array([0., 1., 4., 9., 16., 25.])   # samples of some signal
# Suppose a "new" grid position falls exactly between original index 2 and 3
left_val, right_val = x[2], x[3]
interpolated = (left_val + right_val) / 2
print(f"average of neighbors x[2]={left_val} and x[3]={right_val} -> {interpolated}")

# np.interp(x_new, x_known, y_known) does full linear interpolation --
# useful to sanity-check your own averaging logic against, NOT a drop-in
# replacement (it doesn't do the simple "average of nearest left/right" rule
# the assignment specifically asks you to implement).
t_known = np.arange(6)
y_known = x
t_query = 2.5
print("np.interp reference value:", np.interp(t_query, t_known, y_known))

average of neighbors x[2]=4.0 and x[3]=9.0 -> 6.5
np.interp reference value: 6.5


**Key NumPy tools for building `interpolate_signal(...)` yourself:**
- `np.isnan(arr)` to find which entries are placeholders/missing (if you build
  your scaled array with `np.nan` for unknown positions first, then fill them in).
- `np.flatnonzero` / `np.where` to locate indices of missing vs. known values.
- Simple index arithmetic (`idx-1`, `idx+1`) to grab the nearest known neighbors.
None of these replace the logic you need to write — they're just the primitives.

## 7. Vectorized math: generating sinusoids without loops

NumPy functions like `np.cos`, `np.sin` operate **elementwise** on an entire
array at once. Never write a Python `for` loop to compute a sample-by-sample
formula — vectorize it.

In [8]:
n = np.arange(0, 20)
A, Omega0, phi = 2.0, np.pi/4, np.pi/6

# Vectorized (fast, idiomatic):
x_vec = A * np.cos(Omega0 * n + phi)

# The equivalent loop version (slow, shown ONLY to illustrate what's happening
# under the hood -- avoid this style in your actual submissions):
x_loop = np.array([A * np.cos(Omega0 * ni + phi) for ni in n])

print("all close:", np.allclose(x_vec, x_loop))
print(x_vec[:6])

all close: True
[ 1.73205081  0.51763809 -1.         -1.93185165 -1.73205081 -0.51763809]


**Phase change vs. time shift, conceptually:** a phase change adds a constant
`Δφ` inside the cosine (`cos(Ω₀n + φ + Δφ)`), while a time shift moves `n` itself
(`cos(Ω₀(n - n0) + φ)`). Expanding the shifted version:
`cos(Ω₀n - Ω₀n0 + φ)` shows the shift *acts like* a phase change of `-Ω₀n0` —
**but only when Ω₀n0 is treated modulo 2π**, which is exactly what Assignment 2
asks you to investigate and demonstrate with real signals rather than take on faith.

## 8. Energy and Power — `sum`, `abs`, and `trapezoid`

- **Discrete-time energy:** $E = \sum_n |x[n]|^2$ → a plain `np.sum`.
- **Continuous-time energy:** $E = \int |x(t)|^2\,dt$ → approximate the integral
  numerically since your signal is sampled, using `np.trapezoid` (trapezoidal
  rule; called `np.trapz` in older NumPy versions, now deprecated/removed in
  NumPy 2.x), which needs both the values and the sample spacing.

In [9]:
n = np.arange(-10, 11)
xn = np.where(np.abs(n) <= 5, 1.0, 0.0)   # a simple rectangular pulse

E_discrete = np.sum(np.abs(xn)**2)
P_discrete = np.mean(np.abs(xn)**2)
print("discrete energy:", E_discrete, " power:", P_discrete)

t = np.linspace(-10, 10, 2001)
dt = t[1] - t[0]
xt = np.where(np.abs(t) <= 5, 1.0, 0.0)

# np.trapezoid numerically approximates the integral of |x(t)|^2 dt
E_continuous = np.trapezoid(np.abs(xt)**2, dx=dt)
print("continuous energy (trapezoid approx):", E_continuous)

discrete energy: 11.0  power: 0.5238095238095238
continuous energy (trapezoid approx): 10.009999999999787


## 9. Comparing signals: MSE and equality checks

Assignment 2 explicitly asks you to use **mean-squared error** to compare two
signals (e.g. "shifted-then-something" vs. "phase-changed"). Never compare
floating point arrays with `==` directly.

In [10]:
a = np.array([1.0, 2.0, 3.0, 4.0])
b = np.array([1.0, 2.0, 3.0000001, 4.0])

# WRONG for floats: exact equality almost never holds due to floating point error
print("a == b elementwise:", a == b)

# Mean Squared Error -- exactly what Assignment 2 wants you to compute
mse = np.mean((a - b) ** 2)
print("MSE:", mse)

# For a boolean "are these basically the same" check, use allclose with a tolerance
print("allclose:", np.allclose(a, b, atol=1e-4))

a == b elementwise: [ True  True False  True]
MSE: 2.4999999918171056e-15
allclose: True


## 10. Copy vs. View — a silent bug that WILL bite you

Slicing a NumPy array often returns a **view** (shares memory with the
original), not a copy. If your `time_reverse` or `time_scale` function modifies
an array in place, and that array is a view of the original signal, you'll
corrupt data you didn't mean to touch.

In [11]:
x = np.array([1, 2, 3, 4, 5])
y = x[1:4]          # this is a VIEW, not a copy
y[0] = 999          # modifying y...
print("x got modified too!:", x)   # ...silently changed x as well!

# Reversal with [::-1] is ALSO a view:
z = x[::-1]
print("is z a view of x?", np.shares_memory(x, z))

# Fix: use .copy() whenever you need an independent array
x2 = np.array([1, 2, 3, 4, 5])
safe_reverse = x2[::-1].copy()
safe_reverse[0] = -1
print("original x2 untouched:", x2)
print("safe_reverse         :", safe_reverse)

x got modified too!: [  1 999   3   4   5]
is z a view of x? True
original x2 untouched: [1 2 3 4 5]
safe_reverse         : [-1  4  3  2  1]


**Rule of thumb:** any time a function you write returns a transformed
version of an input array (reversed, shifted, scaled) and the caller might
later modify the result, return `.copy()` unless you're certain no in-place
modification will happen. This has caused real, hard-to-find bugs in past
students' `time_reverse` / even-odd decomposition code, where modifying
`x_e` accidentally also changed `x` or `x_o`.

## Summary table

| Task in your assignments | NumPy tool(s) |
|---|---|
| Build the time/sample axis | `np.arange`, `np.linspace` |
| Time reversal `x(-t)` | slicing with `[::-1]`, `.copy()`, careful axis bookkeeping |
| Even/odd decomposition | elementwise `+`, `-`, scalar division (`(x + x_rev)/2`) |
| Time shift `x[n-n0]` | zero-padding + `np.concatenate`, or direct formula substitution for sinusoids |
| Phase change | elementwise `np.cos`/`np.sin` with modified phase argument |
| Time sub-scaling `x(t/k)` | float→index mapping, `np.round`, bounds checking |
| Interpolating missing samples | manual averaging of neighbor indices |
| Ignoring out-of-range values | boolean masks, `np.where` |
| Energy / Power | `np.sum`, `np.abs`, `np.mean`, `np.trapz` |
| Comparing signals (MSE) | `np.mean((a-b)**2)`, `np.allclose` |
| Avoiding aliasing bugs | `.copy()` vs. views |

This notebook only shows the *mechanism* of each tool on toy examples — applying
them correctly to `x(t)`, `x[n]`, and the specific edge cases in each assignment
is the actual work you need to do yourself.